In [ ]:
# Cell 1
# Mục đích: tạo kết nối BigQuery và các hàm dùng chung cho cả notebook
from google.cloud import bigquery

PROJECT = "bq-learning-510104"
# client là "kênh kết nối"; location phải khớp vùng dataset (US)
client = bigquery.Client(project=PROJECT, location="US")

def mb(n):
    # đổi byte sang MB; trả None nếu n là None để lỗi "đọc byte quá sớm" lộ ra
    return None if n is None else round(n / 1024 / 1024, 2)

def check(sql):
    # dry run: chỉ ước lượng số byte, không chạy thật, không tốn tiền
    cfg = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    job = client.query(sql, job_config=cfg)
    print(f"Se quet: {mb(job.total_bytes_processed)} MB")

def stats(sql, max_mb=100):
    # chạy thật; trần maximum_bytes_billed (nhân 1024*1024 để đổi MB sang byte)
    cfg = bigquery.QueryJobConfig(
        use_query_cache=False,
        maximum_bytes_billed=max_mb * 1024 * 1024,
    )
    job = client.query(sql, job_config=cfg)      # gửi job, không chờ
    df = job.result().to_dataframe()             # chờ xong rồi lấy kết quả
    print(f"processed: {mb(job.total_bytes_processed)} MB | billed: {mb(job.total_bytes_billed)} MB")
    return df

In [ ]:
# Cell 2
# Mục đích: tính bằng SQL "đáp án đúng" để sau này đối chiếu 3 measure DAX
# (SUM OVER = lũy kế YTD, LAG = cùng tháng năm trước)
sql = """
WITH m AS (
  SELECT
    EXTRACT(YEAR  FROM order_date) AS y,
    EXTRACT(MONTH FROM order_date) AS mth,
    SUM(sale_price)                AS rev
  FROM `bq-learning-510104.dbt_dev.fct_net_sales`
  WHERE EXTRACT(YEAR FROM order_date) IN (2024, 2025)
  GROUP BY y, mth
)
SELECT
  y, mth,
  ROUND(rev, 2)                                              AS net_revenue,
  ROUND(SUM(rev) OVER (PARTITION BY y ORDER BY mth), 2)      AS ytd,
  ROUND(LAG(rev) OVER (PARTITION BY mth ORDER BY y), 2)      AS ly,
  ROUND(SAFE_DIVIDE(rev - LAG(rev) OVER (PARTITION BY mth ORDER BY y),
                    LAG(rev) OVER (PARTITION BY mth ORDER BY y)), 4) AS yoy
FROM m
ORDER BY y, mth
"""
check(sql)          # xem số MB trước khi chạy thật
df = stats(sql)     # chạy thật
print(len(df))      # mong đợi 24 (12 tháng × 2 năm)
df

se quet: 0.69 MB
processed: 0.69 MB | billed: 10.0 MB
24


,y,mth,net_revenue,ytd,ly,yoy
0,2024,1,26561.08,26561.08,NaN,NaN
1,2024,2,30548.27,57109.35,NaN,NaN
2,2024,3,30710.47,87819.82,NaN,NaN
3,2024,4,29560.81,117380.63,NaN,NaN
4,2024,5,33826.46,151207.09,NaN,NaN
5,2024,6,37008.43,188215.52,NaN,NaN
6,2024,7,34338.68,222554.20,NaN,NaN
7,2024,8,35682.73,258236.93,NaN,NaN
8,2024,9,36341.06,294577.99,NaN,NaN
9,2024,10,35439.71,330017.70,NaN,NaN


In [14]:
# Cell 3
# Mục đích: tính bằng SQL doanh thu "cùng kỳ" (từ 1/1 đến cùng ngày-tháng với ngày dữ liệu cuối)
# của 2025 và 2026, để đối chiếu với LY sau khi sửa measure
sql = """
WITH lim AS (
  -- ngày dữ liệu cuối cùng trong bảng
  SELECT MAX(order_date) AS last_day
  FROM `bq-learning-510104.dbt_dev.fct_net_sales`
)
SELECT
  EXTRACT(YEAR FROM order_date) AS y,
  -- tổng từ 1/1 đến cùng ngày-tháng
  ROUND(SUM(sale_price), 2) AS rev_same_period,
  -- riêng tháng 10, cũng đến cùng ngày-tháng
  ROUND(SUM(IF(EXTRACT(MONTH FROM order_date) = 10, sale_price, 0)), 2) AS rev_oct_same_period
FROM `bq-learning-510104.dbt_dev.fct_net_sales`
CROSS JOIN lim
WHERE EXTRACT(YEAR FROM order_date) IN (2025, 2026)
  AND FORMAT_DATE('%m%d', order_date) <= FORMAT_DATE('%m%d', lim.last_day)
GROUP BY y
ORDER BY y
"""

check(sql)
df = stats(sql)
print(len(df))   # mong đợi 2
df

se quet: 0.69 MB
processed: 0.69 MB | billed: 10.0 MB
2


,y,rev_same_period,rev_oct_same_period
0,2025,473647.45,37273.31
1,2026,982813.17,180243.98
